# [1교시]

In [52]:
# import shutil

# # 기존에 불완전하게 생성된 폴더 삭제
# shutil.rmtree('./local_qwen_model', ignore_errors=True)
# print("기존 폴더 삭제 완료. 아래 셀을 다시 실행해 모델을 다운로드하세요.")

In [53]:
%pip install tiktoken sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [54]:
import os
import sys
import torch
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

In [55]:
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
local_model_dir = './local_qwen_model'

In [56]:
if not os.path.exists(local_model_dir):
    os.makedirs(local_model_dir, exist_ok=True)
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype = torch.float32)
    _tokenizer.save_pretrained(local_model_dir)
    _model.save_pretrained(local_model_dir)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype = torch.float32,
                                             device_map='cpu',low_cpu_mem_usage=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [57]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    max_length=None,
    temperature=0.1,
    do_sample=True,
    clean_up_tokenization_spaces=False,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

In [58]:
class GraphState(TypedDict):
    question:str
    context:str
    generation:str
    source_info:str

def retrieve(state:GraphState):
    print('--retrieve node --')
    question = state['question']
    docs = retriever.invoke(question)
    contexts,sources = [],[]
    for i, doc in enumerate(docs):
        # 텍스트데이터 포멧팅
        contexts.append(f'[문서 {i+1}] {doc.page_content}')
        # 메타데이터 추출
        filename = doc.metadata.get('filename', 'N/A')
        f_type = doc.metadata.get('file_type', 'N/A')
        cat = doc.metadata.get('category', 'N/A')
        sources.append(f"-문서 {i+1} [파일명:{filename} / 유형 : {f_type} / 카테고리 : {cat} ]")
    contexts_str = '\n\n'.join(contexts)
    sources_str = '\n\n'.join(sources)
    print(f'Retrieve {len(docs)} documents')
    return {"context":contexts_str, "source_info":sources_str}

def generate(state:GraphState):
    print('--generate node --')
    question = state['question']
    context = state['context']
    # LLM을 위한 프롬프트 템플릿
    messages = [
        {'role':'system', 'content':"주어진 문장을 참고해서 사용자의 질문에 한국어로 정확하고 간결하게 답변하세요"},
        {'role':'user','content':f'[본문]\n{context}\n\n[질문]\n{question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)
    return {'generation':response.strip(), "source_info":state["source_info"]}

# [2교시]

In [59]:
workflow = StateGraph(GraphState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)
workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)
app = workflow.compile()

In [60]:
# user_question = input("질문\n")
# inputs = {'question':user_question}
# final_state = app.invoke(inputs)
# print('\n========= 답변 =======')
# print(final_state['generation'])
# print('\n========= 출처정보 =======')
# print(final_state['source_info'])
# print('\n=========================')

In [61]:
final_state.keys()

NameError: name 'final_state' is not defined

In [ ]:
print(final_state['source_info'])

# [3교시]

In [ ]:
# !pip install rouge kiwipiepy lm-eval

In [ ]:
# 정량적 평가 BLEU(기계번역 성능평가) ROUGE
# BLEU 예측 문장과 정답 문장이 얼마나 많은 n-gram을 공유하는지.
# BLEU-1 : unigram
# 단점 : 동의어를 인식 못함 The car is fast / The automobile is quick

# ROUGE(문서요약 평가)
# n-gram을 비교... Recall 중심  The cat sat on the mat / The cat sat
# 정답 : 6개, 예측 : 3개    6 / 3 = 0.5 --> recall  3 / 3 --> 1.0 Precision 
# ROUGE-1 : unigram... 

# 최근에는 다양한 평가지표가 사용 BERTScore... LLM-as-a-judge

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi() # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩 기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram_improved(reference, candidate):
    smothie = SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    bleu_score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str, ref_str)[0]
    return bleu_score, scores

ref = '파리를 여행할 때는 에펠타보가 루브르 박물관을 꼭 방문해야 합니다.'
cand = '파리 여행 시 에펠탑과 루브르 박문관은 반드시 가봐야 할 명소입니다.'

bleu, rouge_rs = evaluate_ngram_improved(ref, cand)

print(f'BLEU : {bleu}')
print(f"ROUGE-1 F1: {rouge_rs['rouge-1']['f']:.4f}")

In [ ]:
from lm_eval.tasks import TaskManager
task_manager = TaskManager()
all_tasks = list(task_manager.task_index.keys())
all_tasks

In [ ]:
[task for task in all_tasks if 'hellaswag' in task]

In [ ]:
# LM-Evaluation_harness 벤치 마크 평가
# 모델의 추론 능력을 벤치마크 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
try:
    result = lm_eval.simple_evaluate(
        model="hf",
        model_args = "pretrained=skt/kogpt2-base-v2,dtype=float32",
        tasks=['hellaswag'],
        device="cpu",
        limit=2
        )
    print("LM-Eval 정확도(acc): {result['results']['hallaswag']['acc, none']}")
except Exception as e:
    print(e)

In [ ]:
# 정답형태가 고정되어 있지않은 생성형 태스크의 경우 상용모델(gpt)을 판단기준으로 활용
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()
def evaluate_with_gpt(prompt, generated_text):
    eval_prompt = f'''
다음 질문과 답변을 보고 정확성, 유창성, 관련성을 기준으로 1~5점 사이의 점수와 이류를 작성해주세요
질문:{prompt}
답변:{generated_text}
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages=[{'role':'user','content':eval_prompt}],
            max_completion_tokens = 150
        )
        return response.choices[0].message.content
    except Exception as e:
        return e
print(evaluate_with_gpt('프랑시의 명소는?','파리 여행시 에팔탑과 루브르 박물관은 반드시 가봐야 할 명소입니다.')    )

# [4교시]

In [ ]:
# BASE모델 , 파인튜닝 Instruct 성능 비교
# Qween2.5  Qween2.5_instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = '프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘'
base_model_id = 'Qwen/Qwen2.5-0.5B'
instruct_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

def generate(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.1,
        do_sample=True,
        max_new_tokens=40
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
print(f'질문 : {prompt}')
base_output = generate(base_model_id)
print(f'[Base모델 출력 : {base_output}]')
instruct_output = generate(instruct_model_id)
print(f'[Instruct 모델 출력 : {instruct_output}]')

In [62]:
# 정량적 평가 BL, ROUGE
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    bleu_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    rouge_f1 = scores['rouge-1']['f']
    return scores,rouge_f1

ref = '파리의 명소로는 에펠탑과 루브르 박물관을 추천합니다.'
b_bleu, b_rouge = evaluate_ngram(ref,base_output)
i_bleu, i_rouge = evaluate_ngram(ref,instruct_output)

print(f'Base모델 BLEU : {b_bleu} ROUGE-1 : {b_rouge}')
print(f'Instruct모델 BLEU : {i_bleu} ROUGE-1 : {i_rouge}')

Base모델 BLEU : {'rouge-1': {'r': 0.38461538461538464, 'p': 0.22727272727272727, 'f': 0.285714281044898}, 'rouge-2': {'r': 0.08333333333333333, 'p': 0.038461538461538464, 'f': 0.05263157462603914}, 'rouge-l': {'r': 0.3076923076923077, 'p': 0.18181818181818182, 'f': 0.22857142390204094}} ROUGE-1 : 0.285714281044898
Instruct모델 BLEU : {'rouge-1': {'r': 0.38461538461538464, 'p': 0.20833333333333334, 'f': 0.27027026571219875}, 'rouge-2': {'r': 0.16666666666666666, 'p': 0.06666666666666667, 'f': 0.09523809115646274}, 'rouge-l': {'r': 0.3076923076923077, 'p': 0.16666666666666666, 'f': 0.21621621165814472}} ROUGE-1 : 0.27027026571219875


In [63]:
# LM-Evaluation-Harness평가
# 모델의 추론 능력을 벤치마크 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
def lm_eval_score(model_id):
    try:
        result = lm_eval.simple_evaluate(
            model="hf",
            model_args = f"pretrained={model_id},dtype=float32",
            tasks=['hellaswag'],
            device="cpu",
            limit=2
        )
        print(f"LM-Eval 정확도(acc): {result['results']['hellaswag']['acc,none']}")
    except Exception as e:
        print(e)
lm_eval_score(base_model_id)
print('='*100)
lm_eval_score(instruct_model_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:03<00:00,  2.05it/s]
pretrained=pretrained=Qwen/Qwen2.5-0.5B-Instruct,dtype=float32 appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).


LM-Eval 정확도(acc): 0.0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:03<00:00,  2.23it/s]


LM-Eval 정확도(acc): 0.0


# [5교시]

In [64]:
# GPT를 활용한 정성평가(LLM-as-a-Judge)
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()

In [65]:
def evaluate_with_gpt(generate_text):
    eval_prompt = f'''
질문:{prompt}
답변:{generate_text}
위 질문에 대한 답변의 정확성, 유창성을 1~5점으로 평가하고 이유를 100자 이내로 작성하세요.
'''
    try:
        response = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = [{'role':'user', 'content':eval_prompt}],
            max_completion_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return e
print('BASE model gpt eval')
print(evaluate_with_gpt(base_output))
print('\nInstruct model gpt eval')
print(evaluate_with_gpt(instruct_output))

BASE model gpt eval
평가: **2/5**  
이유(100자 이내): **루이스 파리 등 명소가 부정확하고, 두 곳만 제시했는지 불명확하며, 문장이 매끄럽지 않음.**

Instruct model gpt eval
평가: 1/5  
이유(100자 이내): 두 명소만 추천하지 못했고, “파리의 중심지인 로마”는 부정확한 표현(로마는 다른 도시). 유창성도 떨어짐.


In [66]:
# 벡터데이터베이스 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [67]:
# 1. 꼬여있는 패키지들을 싹 다 강제 삭제합니다.
# !pip uninstall -y transformers torch torchvision torchaudio trl bitsandbytes

# 2. 호환성이 검증된 안정적인 셋트로 한 번에 재설치합니다.
# !pip install --no-cache-dir torch==2.5.1 "transformers==4.45.2" trl==0.11.0 bitsandbytes accelerate peft datasets scipy

In [68]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16    
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')

Exception in thread Thread-705 (_readerthread):
Traceback (most recent call last):
  File "c:\miniconda\envs\edu_env\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\miniconda\envs\edu_env\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\miniconda\envs\edu_env\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\miniconda\envs\edu_env\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

c:\miniconda\envs\edu_env\lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\miniconda\envs\edu_env\lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [3]:
from datasets import load_dataset
# 범용 비지니스 및 지시어 데이터셋
load_dataset("beomi/KoAlpaca-v1.1a", split='train')

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 21155
})

In [4]:
# 범용 비즈니스 및 지시어 데이터셋 (KoAlpaca) 로드
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
# 모델이 알아들을 수 있도록 Qwen Chat Template 적용 함수
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Respond in Korean."},
        {"role": "user", "content": example["instruction"]}
    ]
    
    # 1. 챗 템플릿 적용
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2.  만약 버그로 인해 리스트(숫자)가 반환되었다면 강제로 문자열로 디코딩
    if isinstance(text, list):
        text = tokenizer.decode(text)
        
    # 3. 정답(output)과 종료 토큰 추가
    text += example["output"] + tokenizer.eos_token
    return {"text": text}
# 실습 시간 단축을 위해 2,000건의 데이터만 샘플링하여 훈련 세트로 구성
train_dataset = dataset.select(range(2000)).map(format_prompt)
print("\n[전처리된 데이터 예시]")
print(train_dataset[0]["text"])


[전처리된 데이터 예시]
<|im_start|>system
You are a helpful AI assistant. Respond in Korean.<|im_end|>
<|im_start|>user
양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?<|im_end|>
<|im_start|>assistant
양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.

 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? 

고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.<|im_end|>


In [5]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# k-bit 양자화된 모델을 파인튜닝할 수 있도록 전처리
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 최신 trl 라이브러리에 맞추어 TrainingArguments 대신 SFTConfig 사용
training_args = SFTConfig(
    output_dir="./qwen-koalpaca-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # 실습용이므로 100 step만 진행
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True, # H100의 BFloat16 가속 활성화
    dataset_text_field="text", # 에러가 났던 인자들을 여기로 이동시켰습니다!
    max_seq_length=512,        # 이것도 여기로 이동
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    peft_config=lora_config
)

max_steps is given, it will override any value given in num_train_epochs


In [6]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.613000
20,2.409000
30,2.314000
40,2.279000
50,2.205700
60,2.221000
70,2.196100
80,2.257000
90,2.230100
100,2.207900


TrainOutput(global_step=100, training_loss=2.2932673835754396, metrics={'train_runtime': 69.577, 'train_samples_per_second': 22.996, 'train_steps_per_second': 1.437, 'total_flos': 1446750388064256.0, 'train_loss': 2.2932673835754396, 'epoch': 0.8})

In [10]:
# 모델 병합
from peft import PeftModel

# del model
# del tainer
# torch.cuda.empty_cache()
# 원본모델을 양자화 없이 FP16으로 로드
base_model =  AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map='auto'
)
# 방금 학습한 LoRA 가중치를 불러와서 update
model = PeftModel.from_pretrained(base_model, './qwen-koalpaca-lora/checkpoint-100')
# 가중치 최종 병합
merged_model = model.merge_and_unload()
# 병합된 모델을 저장
save_directory = "./Qwen-KoAlpaca-Merged"
merged_model.save_pretrained(save_directory)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
c:\miniconda\envs\edu_env\lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for base_model.model.model.layers.21.self_attn.q_proj.lora_A.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\miniconda\envs\edu_env\lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for base_model.model.model.layers.21.self_attn.q_proj.lora_B.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_fro

KeyError: 'base_model.model.model.model.embed_tokens'

In [72]:
# Windows/Mac 로컬 환경 패키지 설치
!pip install huggingface_hub gguf
# Windows C++ 빌드 에러를 방지하기 위해 사전 컴파일된 CPU 전용 버전을 설치합니다.
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

# 모델을 GGUF로 변환해 주는 공식 스크립트를 쓰기 위해 llama.cpp 저장소 다운로드
!git clone https://github.com/ggerganov/llama.cpp.git

  Using cached gguf-0.19.0-py3-none-any.whl.metadata (4.1 kB)
Using cached gguf-0.19.0-py3-none-any.whl (118 kB)
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
     ---------------------------------------- 6.3/6.3 MB 77.8 MB/s  0:00:00
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)

   -------------------- ------------------- 1/2 [llama-cpp-python]
   -------------------- ------------------- 1/2 [llama-cpp-python]
   -------------------- ------------------- 1/2 [llama-cpp-python]
   ---------------------------------------- 2/2 [llama-cpp-python]



Cloning into 'llama.cpp'...
Updating files:  15% (454/2950)
Updating files:  16% (472/2950)
Updating files:  17% (502/2950)
Updating files:  18% (531/2950)
Updating files:  19% (561/2950)
Updating files:  20% (590/2950)
Updating files:  21% (620/2950)
Updating files:  22% (649/2950)
Updating files:  23% (679/2950)
Updating files:  24% (708/2950)
Updating files:  25% (738/2950)
Updating files:  26% (767/2950)
Updating files:  27% (797/2950)
Updating files:  28% (826/2950)
Updating files:  29% (856/2950)
Updating files:  30% (885/2950)
Updating files:  31% (915/2950)
Updating files:  32% (944/2950)
Updating files:  33% (974/2950)
Updating files:  33% (988/2950)
Updating files:  34% (1003/2950)
Updating files:  35% (1033/2950)
Updating files:  36% (1062/2950)
Updating files:  37% (1092/2950)
Updating files:  38% (1121/2950)
Updating files:  39% (1151/2950)
Updating files:  40% (1180/2950)
Updating files:  41% (1210/2950)
Updating files:  42% (1239/2950)
Updating files:  43% (1269/2950)
Up

In [73]:
# 모델을 FP16 정밀도의 GGUF 포맷으로 변환합니다.
!python llama.cpp/convert_hf_to_gguf.py ./Qwen-KoAlpaca-Merged --outtype f16 --outfile qwen_koalpaca_fp16.gguf

INFO:hf-to-gguf:Loading model: Qwen-KoAlpaca-Merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {896, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16 -

# 7교시

In [81]:
# 로컬 CPU 고속 추론 테스트
from llama_cpp import Llama
import sys

# GGUF 모델 로드(GPU가 없어도 CPU 단일 멀티코어를 활용해 고속으로 구동)
llm = Llama(
    model_path='./qwen_koalpaca_fp16.gguf',
    n_ctx=1024,
    n_threads=4,
    verbose=False
)
prompt = '프랑스 파리를 여행하려고 해. 꼭 가봐야 할 명소를 두 곳만 추천해줘'
# 파인튜닝 시 학습했던 Qwen Chat Template 포맷을 그대로 입력합니다.
formatted_prompt = f"<|im_start|>system\nYou are a helpful AI assistant. Respond in Korean.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

print('답변 생성중.....(cpu 추론)')
output = llm(
    formatted_prompt,
    max_tokens=150,
    temperature=0.7,
    stop = ["<|im_end|>"]
)
print('파인튜닝 완료 모델의 답변')
print(output['choices'][0]['text'].strip())

llama_context: n_ctx_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


답변 생성중.....(cpu 추론)
파인튜닝 완료 모델의 답변
미안합니다, 파리를 여행하려고 하기 때문에, 파리의 명소를 추천해드릴 수 없습니다. 파리는 많은 명소를 가지고 있어요. 예를 들어, 파리의 전통적인 요리인 피자로 불리우는 파리스 파리스, 푸드라인, 파리의 전통적인 음식인 파리치우스, 그리고 전통적인 음식인 파리치코르로는 추천해드릴 수 있습니다. 하지만, 파리의 명소를 살펴보는 것은 시간이 좀 걸리는 것 같아서, 이동하기에는 적절할 수 없습니다. 다른 파리의 명
